In [2]:
!pip install transformers datasets accelerate scikit-learn sentencepiece protobuf -q

In [3]:
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/deberta-v3-small.zip"
cache_dir = os.path.expanduser("~/.cache/huggingface/hub/")
os.makedirs(cache_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    for info in z.infolist():
        fixed_name = info.filename.replace("\\", "/")
        out_path = os.path.join(cache_dir, fixed_name)
        if info.is_dir() or fixed_name.endswith("/"):
            os.makedirs(out_path, exist_ok=True)
        elif info.file_size == 0:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
        else:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            with z.open(info) as src, open(out_path, 'wb') as dst:
                dst.write(src.read())

model_cache = os.path.join(cache_dir, "models--microsoft--deberta-v3-small")
print(f"Model extracted to: {model_cache}")
print(f"Contents: {os.listdir(model_cache)}")
snap_dir = os.path.join(model_cache, "snapshots")
if os.path.isdir(snap_dir):
    rev = os.listdir(snap_dir)[0]
    print(f"Snapshot: {rev}")
    print(f"Files: {os.listdir(os.path.join(snap_dir, rev))}")

Mounted at /content/drive
Model extracted to: /root/.cache/huggingface/hub/models--microsoft--deberta-v3-small
Contents: ['snapshots', 'blobs', 'refs']
Snapshot: a36c739020e01763fe789b4b85e2df55d6180012
Files: ['config.json', 'tf_model.h5', 'pytorch_model.bin', '.gitattributes', 'README.md', 'tokenizer_config.json', 'spm.model']


In [4]:
import pandas as pd
import numpy as np
import torch
import json
import os
import gc

os.environ["HF_HUB_OFFLINE"] = "1"

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB


In [5]:
df = pd.read_csv('ExioNAICS_preprocessed.csv')
df['NAICS Code'] = df['NAICS Code'].astype(str)

num_labels = df['label'].nunique()

print(f"Dataset shape: {df.shape}")
print(f"Number of classes: {num_labels}")
print(f"Label range: {df['label'].min()} to {df['label'].max()}")
df.head()

Dataset shape: (20535, 3)
Number of classes: 1115
Label range: 0 to 1114


,clean_description,NAICS Code,label
0,foot locker inc foot locker is a specialty ret...,448150,630
1,cengage is the education and technology compan...,511130,744
2,cardinal scale manufacturing company manufactu...,333997,400
3,alticor inc owns and manages manufacturing and...,454390,661
4,rosboro is north america s largest producer of...,335110,428


# Training Config

In [6]:
SANITY_CHECK = True

SESSION = 1  # 1, 2, or 3

SESSION_CONFIGS = {
    1: {"learning_rate": 2e-5, "batch_size": 16, "weight_decay": 0.01, "name": "conservative"},
    2: {"learning_rate": 3e-5, "batch_size": 32, "weight_decay": 0.01, "name": "fast_convergence"},
    3: {"learning_rate": 1e-5, "batch_size": 16, "weight_decay": 0.1,  "name": "slow_strong_reg"},
}

config = SESSION_CONFIGS[SESSION]

MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LENGTH = 128
NUM_EPOCHS = 50
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 5
N_FOLDS = 1 if SANITY_CHECK else 5
SEED = 42

print(f"=== Session {SESSION}: {config['name']} ===")
print(f"  SANITY CHECK:   {SANITY_CHECK} ({'1 fold only' if SANITY_CHECK else 'full 5-fold'})")
print(f"  Learning rate:  {config['learning_rate']}")
print(f"  Batch size:     {config['batch_size']}")
print(f"  Weight decay:   {config['weight_decay']}")
print(f"  Max length:     {MAX_LENGTH}")
print(f"  Epochs:         {NUM_EPOCHS} (with early stopping, patience={EARLY_STOPPING_PATIENCE})")
print(f"  Folds:          {N_FOLDS}")
print(f"  Warmup ratio:   {WARMUP_RATIO}")

=== Session 1: conservative ===
  SANITY CHECK:   True (1 fold only)
  Learning rate:  2e-05
  Batch size:     16
  Weight decay:   0.01
  Max length:     128
  Epochs:         50 (with early stopping, patience=5)
  Folds:          1
  Warmup ratio:   0.1


In [7]:
from sklearn.model_selection import train_test_split

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if SANITY_CHECK:
    from sklearn.preprocessing import LabelEncoder
    label_counts = df['label'].value_counts()
    valid_labels = label_counts[label_counts >= 2].index
    df_filtered = df[df['label'].isin(valid_labels)].reset_index(drop=True)
    print(f"Filtered: {len(df)} -> {len(df_filtered)} samples ({len(df) - len(df_filtered)} removed with <2 samples)")

    le_refit = LabelEncoder()
    df_filtered['label'] = le_refit.fit_transform(df_filtered['label'])
    print(f"Re-encoded labels: 0 to {df_filtered['label'].max()}")

    train_idx, val_idx = train_test_split(
        np.arange(len(df_filtered)), test_size=0.2, random_state=SEED, stratify=df_filtered['label']
    )
    folds = [(train_idx, val_idx)]
    df = df_filtered
    num_labels = df['label'].nunique()
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    folds = list(skf.split(df['clean_description'], df['label']))

for i, (train_idx, val_idx) in enumerate(folds):
    train_labels = df['label'].iloc[train_idx]
    val_labels = df['label'].iloc[val_idx]
    print(f"Fold {i+1}: train={len(train_idx)}, val={len(val_idx)}, "
          f"train classes={train_labels.nunique()}, val classes={val_labels.nunique()}")

The tokenizer you are loading from 'microsoft/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Filtered: 20535 -> 20533 samples (2 removed with <2 samples)
Re-encoded labels: 0 to 1112
Fold 1: train=16426, val=4107, train classes=1113, val classes=1102


In [8]:
def tokenize_data(texts, labels, tokenizer, max_length):
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors=None,
    )
    dataset = Dataset.from_dict({
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": labels,
    })
    return dataset

print("Tokenization function defined.")
print(f"Using tokenizer: {MODEL_NAME}, max_length: {MAX_LENGTH}")

Tokenization function defined.
Using tokenizer: microsoft/deberta-v3-small, max_length: 128


In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    top1_preds = np.argmax(logits, axis=1)

    top1_acc = accuracy_score(labels, top1_preds)
    macro_f1 = f1_score(labels, top1_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(labels, top1_preds, average='weighted', zero_division=0)

    top5_acc = np.mean([
        1 if label in np.argsort(logit)[-5:] else 0
        for logit, label in zip(logits, labels)
    ])
    top10_acc = np.mean([
        1 if label in np.argsort(logit)[-10:] else 0
        for logit, label in zip(logits, labels)
    ])

    return {
        "top1_accuracy": top1_acc,
        "top5_accuracy": top5_acc,
        "top10_accuracy": top10_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

print("Metrics function defined: Top-1, Top-5, Top-10 accuracy, Macro F1, Weighted F1")

Metrics function defined: Top-1, Top-5, Top-10 accuracy, Macro F1, Weighted F1


In [10]:
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

if hasattr(torch.backends.cuda, 'enable_flash_sdp'):
    torch.backends.cuda.enable_flash_sdp(False)
if hasattr(torch.backends.cuda, 'enable_mem_efficient_sdp'):
    torch.backends.cuda.enable_mem_efficient_sdp(False)

print("Disabled TF32 and flash attention to prevent FP16 CUBLAS errors")

Disabled TF32 and flash attention to prevent FP16 CUBLAS errors


In [11]:
print("=== Model Diagnostic ===")
test_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
test_model.eval()

inputs = tokenizer("this is a test company that manufactures steel", return_tensors="pt")
with torch.no_grad():
    outputs = test_model(**inputs)

print(f"Logits shape: {outputs.logits.shape}")
print(f"Logits sample: {outputs.logits[0, :5]}")
print(f"Any NaN in logits: {torch.isnan(outputs.logits).any().item()}")
print(f"Any Inf in logits: {torch.isinf(outputs.logits).any().item()}")
print(f"Loss: {outputs.loss}")

if torch.isnan(outputs.logits).any() or torch.isinf(outputs.logits).any():
    print("\nDIAGNOSIS: Cached model is BROKEN — producing NaN/Inf on forward pass.")
    print("Solution: Delete cache and download model directly.")
else:
    print("\nDIAGNOSIS: Model forward pass is healthy.")

del test_model
gc.collect()

=== Model Diagnostic ===


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias         

Logits shape: torch.Size([1, 1113])
Logits sample: tensor([-0.0606,  0.0105,  0.2083,  0.0885, -0.0330], dtype=torch.float16)
Any NaN in logits: False
Any Inf in logits: False
Loss: None

DIAGNOSIS: Model forward pass is healthy.


178

In [12]:
WARMUP_HEAD_EPOCHS = 3

all_fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold_idx+1}/{N_FOLDS} — Session {SESSION} ({config['name']})")
    print(f"{'='*60}")

    train_texts = df['clean_description'].iloc[train_idx].tolist()
    val_texts = df['clean_description'].iloc[val_idx].tolist()
    train_labels = df['label'].iloc[train_idx].tolist()
    val_labels = df['label'].iloc[val_idx].tolist()

    train_dataset = tokenize_data(train_texts, train_labels, tokenizer, MAX_LENGTH)
    val_dataset = tokenize_data(val_texts, val_labels, tokenizer, MAX_LENGTH)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
        torch_dtype=torch.float32,
    )

    param = next(model.deberta.parameters())
    print(f"  Model dtype: {param.dtype}")
    print(f"  Backbone weight check — mean: {param.data.mean():.6f}, std: {param.data.std():.6f}")

    # --- Phase 1: Freeze backbone, train only classification head ---
    print(f"\n  Phase 1: Training classification head only ({WARMUP_HEAD_EPOCHS} epochs)...")
    for p in model.deberta.parameters():
        p.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    output_dir = f"./results/session_{SESSION}_fold_{fold_idx+1}"

    phase1_args = TrainingArguments(
        output_dir=output_dir + "_phase1",
        num_train_epochs=WARMUP_HEAD_EPOCHS,
        per_device_train_batch_size=config['batch_size'],
        per_device_eval_batch_size=64,
        learning_rate=1e-3,
        weight_decay=0.01,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        bf16=False,
        fp16=False,
        seed=SEED,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=phase1_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )
    trainer.train()

    phase1_eval = trainer.evaluate()
    print(f"  Phase 1 done — Top-1: {phase1_eval['eval_top1_accuracy']:.4f}, "
          f"Top-5: {phase1_eval['eval_top5_accuracy']:.4f}, Loss: {phase1_eval['eval_loss']:.4f}")

    del trainer
    gc.collect()
    torch.cuda.empty_cache()

    # --- Phase 2: Unfreeze backbone, fine-tune everything ---
    print(f"\n  Phase 2: Fine-tuning full model ({NUM_EPOCHS} epochs)...")
    for p in model.deberta.parameters():
        p.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable params: {trainable:,} / {total:,} (100%)")

    phase2_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=config['batch_size'],
        per_device_eval_batch_size=64,
        learning_rate=config['learning_rate'],
        weight_decay=config['weight_decay'],
        warmup_ratio=WARMUP_RATIO,
        max_grad_norm=1.0,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        logging_steps=50,
        bf16=False,
        fp16=False,
        seed=SEED,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=phase2_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    last_checkpoint = None
    if os.path.isdir(output_dir):
        checkpoints = [os.path.join(output_dir, d) for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
        if checkpoints:
            last_checkpoint = max(checkpoints, key=os.path.getmtime)
            print(f"  Resuming from checkpoint: {last_checkpoint}")

    trainer.train(resume_from_checkpoint=last_checkpoint)

    epoch_log = pd.DataFrame(trainer.state.log_history)
    epoch_log.to_csv(f"results/session_{SESSION}_fold_{fold_idx+1}_epoch_log.csv", index=False)
    print(f"  Epoch log saved to: results/session_{SESSION}_fold_{fold_idx+1}_epoch_log.csv")

    eval_results = trainer.evaluate()
    eval_results["fold"] = fold_idx + 1
    all_fold_results.append(eval_results)

    print(f"\nFold {fold_idx+1} results:")
    print(f"  Top-1 Accuracy: {eval_results['eval_top1_accuracy']:.4f}")
    print(f"  Top-5 Accuracy: {eval_results['eval_top5_accuracy']:.4f}")
    print(f"  Top-10 Accuracy: {eval_results['eval_top10_accuracy']:.4f}")
    print(f"  Macro F1:       {eval_results['eval_macro_f1']:.4f}")
    print(f"  Weighted F1:    {eval_results['eval_weighted_f1']:.4f}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"  ALL FOLDS COMPLETE — Session {SESSION}")
print(f"{'='*60}")


  FOLD 1/1 — Session 1 (conservative)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias         

  Model dtype: torch.float32
  Backbone weight check — mean: -0.016238, std: 0.063136

  Phase 1: Training classification head only (3 epochs)...
  Trainable params: 1,446,489 / 142,750,809 (1.0%)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch log saved to: results/session_1_fold_1_epoch_log.csv



Fold 1 results:
  Top-1 Accuracy: 0.0903
  Top-5 Accuracy: 0.2459
  Top-10 Accuracy: 0.3158
  Macro F1:       0.0783
  Weighted F1:    0.0832

  ALL FOLDS COMPLETE — Session 1


In [13]:
metrics_keys = ["eval_top1_accuracy", "eval_top5_accuracy", "eval_top10_accuracy",
                "eval_macro_f1", "eval_weighted_f1"]

print(f"\n=== Session {SESSION} ({config['name']}) — Summary ===\n")
print(f"{'Metric':<22} {'Mean':>8} {'Std':>8}  Per-fold values")
print("-" * 75)

summary = {"session": SESSION, "config": config}
for key in metrics_keys:
    values = [r[key] for r in all_fold_results]
    mean_val = np.mean(values)
    std_val = np.std(values)
    fold_str = ", ".join([f"{v:.4f}" for v in values])
    print(f"{key:<22} {mean_val:>8.4f} {std_val:>8.4f}  [{fold_str}]")
    summary[key] = {"mean": mean_val, "std": std_val, "per_fold": values}

os.makedirs("results", exist_ok=True)
results_file = f"results/session_{SESSION}_{config['name']}.json"
with open(results_file, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nResults saved to: {results_file}")


=== Session 1 (conservative) — Summary ===

Metric                     Mean      Std  Per-fold values
---------------------------------------------------------------------------
eval_top1_accuracy       0.0903   0.0000  [0.0903]
eval_top5_accuracy       0.2459   0.0000  [0.2459]
eval_top10_accuracy      0.3158   0.0000  [0.3158]
eval_macro_f1            0.0783   0.0000  [0.0783]
eval_weighted_f1         0.0832   0.0000  [0.0832]

Results saved to: results/session_1_conservative.json


In [16]:
import zipfile, glob

with zipfile.ZipFile("results.zip", "w") as zf:
    for f in glob.glob("results/*.json") + glob.glob("results/*.csv"):
        zf.write(f)
        print(f"  Added: {f}")

from google.colab import files
files.download("results.zip")
print("\nDownloaded results.zip (JSON + CSV files only)")

  Added: results/session_1_conservative.json
  Added: results/session_1_fold_1_epoch_log.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Downloaded results.zip (JSON + CSV files only)
